# CWT + SST Deep Dive

This notebook explores the Continuous Wavelet Transform and Synchrosqueezed CWT in detail.

Topics covered:
1. Pure-tone CWT energy profile — verifying scale-frequency convention
2. CWT vs SST concentration comparison (Gini coefficient)
3. Gamma threshold effect — how `gamma` controls the hard-thresholding
4. Voice density (`nv`) effect on frequency resolution
5. Two-chirp separation: CWT vs SST vs MSST
6. Ridge extraction and reconstruction fidelity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import wavesst
from wavesst.viz.tf_plot import _to_numpy
from wavesst.viz.interactive import iplot_cwt, iplot_sst, iplot_ridges, iplot_components

%matplotlib widget
print(f'wavesst {wavesst.__version__}')

## 1 — Pure-Tone CWT Energy Profile

For a single pure tone at `f0`, the CWT `|W(a, t)|²` should peak at the scale `a` corresponding to `f0`.  Plotting the marginal `Σ_t |W(a, t)|²` vs frequency confirms the scale-to-frequency mapping.

In [ ]:
FS = 256.0
N  = 1024
t  = np.arange(N) / FS
cfg = wavesst.Config(device='cpu', dtype='complex64')

f0 = 40.0
x  = np.cos(2 * np.pi * f0 * t).astype(np.float32)

cwt_result = wavesst.cwt(x, fs=FS, nv=32, cfg=cfg)
energy     = cwt_result.W.abs().pow(2).sum(dim=1).numpy()  # (n_scales,)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(cwt_result.freqs, energy)
ax.axvline(f0, color='red', linestyle='--', label=f'True f₀ = {f0} Hz')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Energy')
ax.set_title('CWT Marginal Energy — Pure Tone')
ax.legend()
plt.tight_layout()
plt.show()

## 2 — CWT vs SST Concentration

SST sharpens the TF representation.  We quantify this with the **Gini coefficient** of the energy profile: higher = more concentrated.

In [ ]:
def gini(e):
    e = np.sort(np.abs(e))
    n = len(e)
    return (2 * np.sum((np.arange(1, n+1) * e)) / (n * e.sum()) - (n+1)/n)

sst_result = wavesst.sst(x, fs=FS, nv=32, gamma='auto', cfg=cfg)

cwt_energy = cwt_result.W.abs().pow(2).sum(dim=1).numpy()
sst_energy = sst_result.Tx.abs().pow(2).sum(dim=1).numpy()

print(f'CWT Gini: {gini(cwt_energy):.4f}')
print(f'SST Gini: {gini(sst_energy):.4f}  (higher = more concentrated)')

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(cwt_result.freqs, cwt_energy)
axes[0].axvline(f0, color='red', linestyle='--')
axes[0].set_title(f'CWT marginal  (Gini={gini(cwt_energy):.3f})')
axes[0].set_xlabel('Frequency (Hz)')
axes[1].plot(sst_result.freqs, sst_energy)
axes[1].axvline(f0, color='red', linestyle='--')
axes[1].set_title(f'SST marginal  (Gini={gini(sst_energy):.3f})')
axes[1].set_xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

## 3 — Gamma Threshold Effect

`gamma` is the hard-threshold applied to `|W(a,t)|` before computing the IF estimator.  Too low: noisy IF estimates included.  Too high: valid signal bins suppressed.

In [ ]:
gammas = [None, 1e-4, 1e-3, 0.01, 0.1]
fig, axes = plt.subplots(1, len(gammas), figsize=(16, 3))

for ax, g in zip(axes, gammas):
    r = wavesst.sst(x, fs=FS, nv=32, gamma=g, cfg=cfg)
    Tx_mag = r.Tx.abs().numpy()
    ax.pcolormesh(r.times, r.freqs, Tx_mag, shading='auto', cmap='inferno')
    ax.set_ylim(0, 80)
    ax.set_title(f'γ = {"auto" if g is None else g}')
    ax.set_xlabel('Time (s)')
    if ax is axes[0]:
        ax.set_ylabel('Frequency (Hz)')

plt.suptitle('SST — Effect of Gamma Threshold', y=1.02)
plt.tight_layout()
plt.show()

## 4 — Voice Density (`nv`) Effect

`nv` controls how many scales are computed per octave.  Higher `nv` → finer frequency resolution but more computation.

In [ ]:
nvs = [8, 16, 32, 64]
fig, axes = plt.subplots(1, len(nvs), figsize=(16, 3))

for ax, nv in zip(axes, nvs):
    r = wavesst.sst(x, fs=FS, nv=nv, gamma='auto', cfg=cfg)
    Tx_mag = r.Tx.abs().numpy()
    ax.pcolormesh(r.times, r.freqs, Tx_mag, shading='auto', cmap='inferno')
    ax.set_ylim(0, 80)
    ax.set_title(f'nv = {nv}  ({r.freqs.shape[0]} freqs)')
    ax.set_xlabel('Time (s)')
    if ax is axes[0]:
        ax.set_ylabel('Frequency (Hz)')

plt.suptitle('SST — Voice Density (nv)', y=1.02)
plt.tight_layout()
plt.show()

## 5 — Two-Chirp: CWT vs SST vs MSST

Side-by-side static comparison, then interactive plots for deeper exploration.

In [ ]:
T2   = 2.0
t2   = np.arange(int(T2 * FS)) / FS
f1   = 20 + 80 * t2 / T2
f2   = 120 - 90 * (t2 / T2) ** 2
x2   = (np.cos(2*np.pi*np.cumsum(f1)/FS) + np.cos(2*np.pi*np.cumsum(f2)/FS)).astype(np.float32)

cwt2  = wavesst.cwt(x2, fs=FS, nv=32, cfg=cfg)
sst2  = wavesst.sst(x2, fs=FS, nv=32, gamma='auto', cfg=cfg)
msst2 = wavesst.msst(x2, fs=FS, nv=32, n_iter=3, gamma='auto', cfg=cfg)
print('CWT, SST, MSST computed.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
datasets = [
    (cwt2.W,    cwt2.times,  cwt2.freqs,  'CWT'),
    (sst2.Tx,   sst2.times,  sst2.freqs,  'SST'),
    (msst2.Tx,  msst2.times, msst2.freqs, 'MSST (3 iter)'),
]
for ax, (tensor, times, freqs, title) in zip(axes, datasets):
    ax.pcolormesh(_to_numpy(times), _to_numpy(freqs),
                  _to_numpy(tensor.abs()), shading='auto', cmap='inferno')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
print('--- Interactive CWT ---')
iplot_cwt(cwt2)

In [ ]:
print('--- Interactive SST ---')
iplot_sst(sst2)

In [ ]:
print('--- Interactive MSST (3 iter) ---')
iplot_sst(msst2)

## 6 — Ridge Extraction and Reconstruction Fidelity

In [ ]:
ridges2 = wavesst.extract_ridges(sst2, n=2, penalty=2.0)
for i, r in enumerate(ridges2):
    print(f'Ridge {i+1}: median = {np.median(r.freq_path):.1f} Hz')
iplot_ridges(sst2, ridges2)

In [ ]:
comps2 = wavesst.reconstruct(sst2, ridges2)
iplot_components(comps2, sst2.times)